<a href="https://colab.research.google.com/github/sichaoj/DS_coding/blob/main/pdf2cdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF to CDF in Python

The CDF is $F(x)=P(X\le x)$.

*   For continuous random variables, integrate the PDF
*   For discrete random variables, add the PMF

| Input | Method | Python tool |
|---|---|---|
| Sampled continuous PDF | Cumulative trapezoidal integration | `scipy.integrate.cumulative_trapezoid` |
| Normal PDF function | Numerical integration | `scipy.integrate.quad` or `scipy.stats.norm.cdf` |
| Discrete PMF | Cumulative sum | `numpy.cumsum` |

## Q1. Given an array of PDF values, calculate the CDF

`x = [0, 1, 2, 3, 4]`, `pdf = [0, 0.2, 0.6, 0.2, 0]`.

| x | PDF | CDF |
|---:|---:|---:|
| 0 | 0 | 0 |
| 1 | 0.2 | 0.1 |
| 2 | 0.6 | 0.5 |
| 3 | 0.2 | 0.9 |
| 4 | 0 | 1.0 |

### 1.1 Mathematical derivation: trapezoidal rule

For a continuous random variable,

$$F(x_i)=\int_{-\infty}^{x_i} f(t)\,dt.$$

Assume that the supplied grid starts at the lower support boundary (or the omitted left-tail probability is negligible). Then $F(x_0)=0$. Approximate the density between adjacent grid points by a straight line. The area of each trapezoid is

$$A_i=\frac{f(x_{i-1})+f(x_i)}{2}(x_i-x_{i-1}).$$

Therefore,

$$F(x_i)\approx F(x_{i-1})+A_i.$$

### Method 1: Pure Python (no imports)

In [ ]:
# array

def pdf_to_cdf(x, pdf):
    if not x or len(x) != len(pdf):
        raise ValueError("x and pdf must have the same nonzero length")
    if any(p < 0 for p in pdf):
        raise ValueError("Density values must be nonnegative")

    cdf = [0]
    for i in range(1, len(x)):
        dx = x[i] - x[i - 1]
        area = (pdf[i - 1] + pdf[i]) / 2 * dx
        cdf.append(cdf[-1] + area)
    return cdf

x = [0, 1, 2, 3, 4]
pdf = [0, 0.2, 0.6, 0.2, 0]
print(pdf_to_cdf(x, pdf))

[0, 0.1, 0.5, 0.9, 1.0]


In [ ]:
# df

import pandas as pd

df = pd.DataFrame({
    'x': [0, 1, 2, 3, 4],
    'pdf': [0, 0.2, 0.6, 0.2, 0]
})

def pdf_to_cdf(df):
    x = df['x'].tolist()
    pdf = df['pdf'].tolist()
    cdf = [0]

    for i in range(1, len(x)):
        dx = x[i] - x[i-1]
        area = (pdf[i-1] + pdf[i]) / 2 * dx
        cdf.append(cdf[-1] + area)

    df['cdf'] = cdf
    return df

print(pdf_to_cdf(df))

   x  pdf  cdf
0  0  0.0  0.0
1  1  0.2  0.1
2  2  0.6  0.5
3  3  0.2  0.9
4  4  0.0  1.0


### Method 2: SciPy method

`cumulative_trapezoid` **syntax:** `cumulative_trapezoid(pdf, x-coor, initial=0)`

In [ ]:
# array
from scipy.integrate import cumulative_trapezoid

cdf = cumulative_trapezoid(pdf, x, initial=0)
print(cdf)

[0.  0.1 0.5 0.9 1. ]


In [ ]:
#df
df['cdf'] = cumulative_trapezoid(df['pdf'], df['x'], initial=0)

print(df)

   x  pdf  cdf
0  0  0.0  0.0
1  1  0.2  0.1
2  2  0.6  0.5
3  3  0.2  0.9
4  4  0.0  1.0


## Q2. Given a discrete PMF, calculate the CDF


Example: $X\in\{0,1,2,3\}$ with PMF $[0.1,0.2,0.4,0.3]$. The CDF is $[0.1,0.3,0.7,1.0]$.



![PMF vs PDF](https://raw.githubusercontent.com/sichaoj/DS_coding/main/images/pdf_vs_pmf.png)

In [ ]:
# array
def pmf_to_cdf(pmf):
    cdf = []
    total = 0
    for p in pmf:
      total += p
      cdf.append(round(total, 2))
    return cdf

x = [0, 1, 2, 3]
pmf = [0.1, 0.2, 0.4, 0.3]
print(pmf_to_cdf(pmf))

[0.1, 0.3, 0.7, 1.0]


In [ ]:
# df

df = pd.DataFrame({
    'x': [0, 1, 2, 3, 4],
    'pmf': [0.1, 0.2, 0.05, 0.3, 0.35]
})

df['cdf'] = df['pmf'].cumsum()
print(df)

   x   pmf   cdf
0  0  0.10  0.10
1  1  0.20  0.30
2  2  0.05  0.35
3  3  0.30  0.65
4  4  0.35  1.00


## Q3. Given a normal PDF, calculate the CDF at x



### Syntax:

```
norm.cdf(x, loc=0, scale=1)
```


*   x: point at which to calculate CDF
*   loc: mean
*   scale: standard deviation




In [ ]:
print(norm.cdf([-1, 0, 1], loc=0, scale=1))

[0.15865525 0.5        0.84134475]
